In [1]:
import pandas as pd
import random
from transformers import AutoTokenizer
from utils.load import LoadDataCSV
from utils.embeddings import Embeddings

# Load datasets

df_loader=LoadDataCSV()


/Users/santiagolares/Projects/semeval_2025/tformervenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#  Data Preparation

In [2]:
fact_checks_mono_sample_1000=pd.read_csv('data/fact_checks_mono_sample_1000.csv')
posts_mono_sample_1000=pd.read_csv('data/posts_train_mono_sample_1000.csv')

embeddings_utils=Embeddings

embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_eng_embedding', 'claim_title_embedding']

fact_checks_mono_sample_1000=embeddings_utils.restore_embeddings(fact_checks_mono_sample_1000, embedding_columns_fact_checks)
posts_mono_sample_1000=embeddings_utils.restore_embeddings(posts_mono_sample_1000, embedding_columns_posts)

In [3]:

# Load your data

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")


In [4]:
# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000['post_id'])]

In [5]:
# Generate extremely small samples
posts_mono_sample_1000_10=posts_mono_sample_1000.sample(n=10)

# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000_10=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000_10['post_id'])]

fact_checks_mono_sample_1000_10=fact_checks_mono_sample_1000[fact_checks_mono_sample_1000['fact_check_id'].isin(mapping_sample_1000_10['fact_check_id'])]


In [6]:

def split_into_chunks(text, max_tokens=512):
    """
    Divide un texto en chunks de hasta max_tokens tokens.
    """
    if not text or not isinstance(text, str):  # Verifica que sea una cadena de texto
        return []
    tokens = tokenizer.encode(text, truncation=True)
    chunks = [tokens[i:i + max_tokens] for i in range(0, len(tokens), max_tokens)]
    return [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

# Chunk posts and fact-check texts
posts_mono_sample_1000["chunks"] = posts_mono_sample_1000["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000["chunks"] = fact_checks_mono_sample_1000["claim_title"].apply(split_into_chunks)

# Prepare positive and negative pairs
import random

def create_pairs(pairs_df, neg_ratio=5):
    """
    Creates positive and negative pairs for contrastive learning.
    """
    positive_pairs = []
    negative_pairs = []

    all_fact_check_ids = set(pairs_df["fact_check_id"].tolist())
    for _, row in pairs_df.iterrows():
        # Positive pair
        positive_pairs.append((row["post_id"], row["fact_check_id"], 1))
        
        # Negative pairs
        possible_negatives = list(all_fact_check_ids - {row["fact_check_id"]})
        if len(possible_negatives) < neg_ratio:
            neg_ratio = len(possible_negatives)  # Adjust if not enough negatives
        negatives = random.sample(possible_negatives, neg_ratio)
        for neg in negatives:
            negative_pairs.append((row["post_id"], neg, 0))
    
    return positive_pairs + negative_pairs

pairs_data = create_pairs(mapping_sample_1000)


In [13]:
posts_mono_sample_1000_10["chunks"] = posts_mono_sample_1000_10["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000_10["chunks"] = fact_checks_mono_sample_1000_10["claim_title"].apply(split_into_chunks)

pairs_data_10 = create_pairs(mapping_sample_1000_10)

/var/folders/dj/58tclkkj76sf6w2h7x_9cyt40000gn/T/ipykernel_78163/3193510769.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_checks_mono_sample_1000_10["chunks"] = fact_checks_mono_sample_1000_10["claim_title"].apply(split_into_chunks)


# Fine-Tuning Dataset Class

In [53]:
import torch
from torch.utils.data import Dataset
import random

class FineTuneContrastiveDataset(Dataset):
    def __init__(self, pairs_data, posts, fact_checks, tokenizer, max_tokens=512):
        """
        pairs_data: Lista de tuplas (post_id, fact_check_id, label).
        posts: DataFrame con los textos asociados a los post_id.
        fact_checks: DataFrame con los textos asociados a los fact_check_id.
        tokenizer: Tokenizador de Hugging Face.
        max_tokens: Máxima longitud permitida de los textos.
        """
        self.pairs_data = pairs_data
        self.posts = posts.set_index("post_id")
        self.fact_checks = fact_checks.set_index("fact_check_id")
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.fact_check_ids = list(self.fact_checks.index)  # Para seleccionar negativos

    def __len__(self):
        return len(self.pairs_data)

    def __getitem__(self, idx):
        # Recupera datos del par original
        post_id, fact_check_id, label = self.pairs_data[idx]
        
        # Texto del post (anchor)
        post_text = " ".join(self.posts.loc[post_id]["chunks"])

        # Texto del fact-check positivo (positive)
        fact_check_text_positive = " ".join(self.fact_checks.loc[fact_check_id]["chunks"])

        # Genera un fact-check negativo (negative)
        negative_fact_check_id = random.choice(
            [fc_id for fc_id in self.fact_check_ids if fc_id != fact_check_id]
        )
        fact_check_text_negative = " ".join(self.fact_checks.loc[negative_fact_check_id]["chunks"])

        # Tokenización
        inputs_anchor = self.tokenizer(post_text, 
                                       truncation=True, max_length=self.max_tokens, 
                                       padding="max_length", return_tensors="pt")
        inputs_positive = self.tokenizer(fact_check_text_positive, 
                                         truncation=True, max_length=self.max_tokens, 
                                         padding="max_length", return_tensors="pt")
        inputs_negative = self.tokenizer(fact_check_text_negative, 
                                         truncation=True, max_length=self.max_tokens, 
                                         padding="max_length", return_tensors="pt")

        return {
            "anchor_input_ids": inputs_anchor["input_ids"].squeeze(0),
            "anchor_attention_mask": inputs_anchor["attention_mask"].squeeze(0),
            "positive_input_ids": inputs_positive["input_ids"].squeeze(0),
            "positive_attention_mask": inputs_positive["attention_mask"].squeeze(0),
            "negative_input_ids": inputs_negative["input_ids"].squeeze(0),
            "negative_attention_mask": inputs_negative["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.float),
        }

# Initialize dataset
dataset = FineTuneContrastiveDataset(pairs_data, posts_mono_sample_1000, fact_checks_mono_sample_1000, tokenizer)


In [37]:
example = dataset[0]
print(f"Input IDs A: {example['input_ids_a'].shape}")
print(f"Attention Mask A: {example['attention_mask_a'].shape}")
print(f"Input IDs B: {example['input_ids_b'].shape}")
print(f"Attention Mask B: {example['attention_mask_b'].shape}")
print(f"Label: {example['label']}")


Input IDs A: torch.Size([512])
Attention Mask A: torch.Size([512])
Input IDs B: torch.Size([512])
Attention Mask B: torch.Size([512])
Label: 1.0


In [38]:
dataset_10 = FineTuneContrastiveDataset(pairs_data_10, posts_mono_sample_1000_10, fact_checks_mono_sample_1000_10, tokenizer)

# Model Fine-Tuning

In [12]:
from transformers import AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class E5FineTuner(nn.Module):
    def __init__(self, model_name, embedding_dim=128):
        """
        Fine-tuneable model for contrastive learning based on pre-trained models.
        Args:
            model_name (str): Name of the pre-trained model from Hugging Face.
            embedding_dim (int): Dimensionality of the projected embedding space.
        """
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.projection = nn.Linear(self.model.config.hidden_size, embedding_dim)  # Projection layer
        self.normalize = nn.LayerNorm(embedding_dim)  # Optional: Normalization for stability

    def forward(self, input_ids, attention_mask):
        """
        Forward pass for encoding input text.
        Args:
            input_ids (torch.Tensor): Tokenized input IDs.
            attention_mask (torch.Tensor): Attention masks for the input IDs.
        Returns:
            torch.Tensor: Normalized embeddings of shape (batch_size, embedding_dim).
        """
        # Get outputs from the transformer
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use pooler_output if available, else mean-pooling over last hidden state
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            embeddings = outputs.pooler_output
        else:
            # Mean-pooling
            embeddings = torch.mean(outputs.last_hidden_state, dim=1)
        
        # Apply projection and normalization
        embeddings = self.projection(embeddings)
        embeddings = self.normalize(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)  # L2-normalization for cosine similarity
        
        return embeddings


# Loss Function

In [11]:
import torch
import torch.nn.functional as F

def contrastive_loss(embeddings, positive_indices, temperature=0.1):
    """
    Compute contrastive loss using InfoNCE formula.

    Args:
        embeddings (torch.Tensor): Tensor of shape (N, D) where N is the batch size and D is the embedding size.
        positive_indices (torch.Tensor): Tensor of shape (N,) indicating the positive pair for each sample.
        temperature (float): Temperature scaling parameter.

    Returns:
        torch.Tensor: Scalar loss.
    """
    # Compute pairwise cosine similarity
    similarity_matrix = F.cosine_similarity(embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=-1)

    # Scale by temperature
    similarity_matrix /= temperature

    # Create labels for positives
    labels = torch.arange(embeddings.size(0)).to(embeddings.device)
    
    # Use cross-entropy loss
    loss = F.cross_entropy(similarity_matrix, labels)
    
    return loss

# Training Loop

In [59]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from torch.utils.tensorboard import SummaryWriter
from torch.nn.functional import cosine_similarity

# Inicializa el escritor de TensorBoard
writer = SummaryWriter(log_dir='./runs/e5_fine_tune_infonce')

torch.mps.empty_cache()

# Hyperparameters
batch_size = 16
epochs = 3
learning_rate = 1e-5
checkpoint_dir = "./checkpoints"

# Crear directorio para checkpoints si no existe
os.makedirs(checkpoint_dir, exist_ok=True)

# Prepare DataLoader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize Model and Optimizer
model = E5FineTuner("intfloat/multilingual-e5-small")
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Scheduler
num_training_steps = len(dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warm-up steps
scheduler = get_scheduler("linear", 
                           optimizer=optimizer, 
                           num_warmup_steps=num_warmup_steps, 
                           num_training_steps=num_training_steps)

# Contrasting loss parameters
temperature = 0.05  # Scaling factor for logits

# Training Loop
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(dataset):
        # Mover los datos al dispositivo
        anchor_input_ids = batch["anchor_input_ids"].to(device)
        anchor_attention_mask = batch["anchor_attention_mask"].to(device)
        positive_input_ids = batch["positive_input_ids"].to(device)
        positive_attention_mask = batch["positive_attention_mask"].to(device)
        negative_input_ids = batch["negative_input_ids"].to(device)
        negative_attention_mask = batch["negative_attention_mask"].to(device)

        anchor_input_ids = anchor_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        anchor_attention_mask = anchor_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch
        positive_input_ids = positive_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        positive_attention_mask = positive_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch
        negative_input_ids = negative_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        negative_attention_mask = negative_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch


        optimizer.zero_grad()

        # Generar embeddings
        anchor_embeddings = model.forward(anchor_input_ids, anchor_attention_mask)
        positive_embeddings = model.forward(positive_input_ids, positive_attention_mask)
        negative_embeddings = model.forward(negative_input_ids, negative_attention_mask)

        # Calcular las similitudes coseno
        pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
        neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)

        # Calcular logits
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1) / temperature
        labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)  # La clase positiva es siempre la primera

        # Calcular la pérdida InfoNCE
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Actualizar métricas
        epoch_loss += loss.item()
        writer.add_scalar('Loss/train', loss.item(), epoch * len(dataloader) + step)
        writer.add_scalar('Learning Rate', scheduler.get_lr()[0], epoch * len(dataloader) + step)

        del anchor_input_ids, anchor_attention_mask, positive_input_ids, positive_attention_mask
        del negative_input_ids, negative_attention_mask, anchor_embeddings, positive_embeddings, negative_embeddings
        torch.mps.empty_cache()

        if step % 10 == 0:
            print(f"Epoch {epoch}, Step {step}, Loss: {loss.item()}")

    # Guardar checkpoint al final de cada época
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': epoch_loss / len(dataloader),
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

    # Registro de métricas de la época
    writer.add_scalar('Loss/epoch', epoch_loss / len(dataloader), epoch)

writer.close()

/Users/santiagolares/Projects/semeval_2025/tformervenv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:382: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


Epoch 0, Step 0, Loss: 0.37095898389816284
Epoch 0, Step 10, Loss: 0.41781505942344666
Epoch 0, Step 20, Loss: 0.26218053698539734
Epoch 0, Step 30, Loss: 0.19574211537837982
Epoch 0, Step 40, Loss: 0.7008527517318726
Epoch 0, Step 50, Loss: 0.23872098326683044
Epoch 0, Step 60, Loss: 0.1983080357313156
Epoch 0, Step 70, Loss: 0.5544043183326721
Epoch 0, Step 80, Loss: 0.22079695761203766
Epoch 0, Step 90, Loss: 0.12156429886817932
Epoch 0, Step 100, Loss: 0.24420151114463806
Epoch 0, Step 110, Loss: 0.27295762300491333
Epoch 0, Step 120, Loss: 0.008397625759243965
Epoch 0, Step 130, Loss: 0.9354369640350342
Epoch 0, Step 140, Loss: 0.07319293916225433
Epoch 0, Step 150, Loss: 0.01259244792163372
Epoch 0, Step 160, Loss: 0.09620831161737442
Epoch 0, Step 170, Loss: 0.7021663188934326
Epoch 0, Step 180, Loss: 0.032121844589710236
Epoch 0, Step 190, Loss: 0.010928775183856487
Epoch 0, Step 200, Loss: 0.0026312987320125103
Epoch 0, Step 210, Loss: 0.011890840716660023
Epoch 0, Step 220, L

KeyboardInterrupt: 

In [ ]:
# Guardar el modelo completo (incluyendo configuraciones y optimizador)
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'loss': loss,
}, "fine_tuned_model_e5_small_1000_1e_5.pth")

In [ ]:
import torch

# Ruta al archivo guardado
checkpoint_path = "fine_tuned_model_e5_small_1000_1e_5.pth"

# Cargar checkpoint
checkpoint = torch.load(checkpoint_path)

# Crear una nueva instancia del modelo y cargar el estado
model_loaded = E5FineTuner("intfloat/multilingual-e5-small")  # Usa la misma configuración que el modelo entrenado
model_loaded.load_state_dict(checkpoint['model_state_dict'])

# Cargar el optimizador si planeas continuar el entrenamiento
optimizer = AdamW(model_loaded.parameters(), lr=1e-5)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Cargar el scheduler si lo usaste
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=0)  # Configuración ajustada
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# Opcional: verifica la pérdida o la época desde donde cargar
epoch = checkpoint['epoch']
loss = checkpoint['loss']

print(f"Modelo cargado desde la época {epoch} con pérdida {loss:.4f}")

# Configurar el dispositivo
device = "mps" if torch.backends.mps.is_available() else "cpu"
model_loaded.to(device)

# Verificar que el modelo funcione correctamente
model_loaded.eval()  # Configura el modelo para inferencia
sample_text = "This is a sample text to verify embeddings."
inputs = tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True).to(device)

# Generar embeddings como prueba
with torch.no_grad():
    embeddings = model_loaded(inputs["input_ids"], inputs["attention_mask"])
    print("Embeddings generados correctamente.")
    print(embeddings.shape)


/var/folders/dj/58tclkkj76sf6w2h7x_9cyt40000gn/T/ipykernel_33888/275585474.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)
/User

Modelo cargado desde la época 2 con pérdida 0.4678
Embeddings generados correctamente.
torch.Size([1, 128])
